# Evaluasi validation bersama (W10)

Satu notebook yang menjalankan **Popularity, MF, TF-IDF, dan Embedding** dengan:
- daftar evaluable users yang sama (`build_protocol` dari `src/protocol.py`),
- fungsi hitung yang sama (`evaluate_model` dari `src/evaluate.py`),
- hasil disimpan ke `outputs/metrics/validation_results.csv`.

Hanya data **train** dan **validation** yang dipakai. File test tidak dibuka.

In [2]:
import sys, json, random, time, datetime, traceback
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.protocol import build_protocol, protocol_counts, compare_with_official
from src.evaluate import evaluate_model, sanity_check_metrics
from src.leakage_check import (LeakageError, run_static_checks,
                               check_recommendations)
from src.popularity import PopularityRecommender
from src.matrix_factorization import MatrixFactorizationRecommender
from src.content_based import TfidfContentRecommender
from src.semantic_based import SemanticEmbeddingRecommender

SEED, K = 42, 10
EXPECTED_USERS = 11659          # jumlah evaluable users resmi (file dosen)
DATA_DIR = str(ROOT / 'data') + '/'
METRICS_DIR = ROOT / 'outputs' / 'metrics'
LOGS_DIR = ROOT / 'outputs' / 'logs'
random.seed(SEED); np.random.seed(SEED)

D:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1. Muat protokol

`build_protocol` mengunci siapa yang dievaluasi (evaluable users) dan item relevan
(rating >= 4 di validation, hanya item kandidat warm). Semua model memakai `relevant` dan `seen` yang sama.

In [4]:
train, val, cand_set, seen, relevant = build_protocol(DATA_DIR)
counts = protocol_counts(train, val, cand_set, seen, relevant)
for key, value in counts.items():
    print(f'{key}: {value}')

official = compare_with_official(relevant, DATA_DIR)
print('Cek vs file resmi:', official)

n_users = len(relevant)
print(f'\nEvaluable users: {n_users:,}')
assert n_users == EXPECTED_USERS, (
    f'Jumlah evaluable users {n_users} tidak sama dengan {EXPECTED_USERS}. '
    'Cek build_protocol dan file data sebelum lanjut.')

train_interactions: 427957
validation_interactions: 41284
train_users: 54483
train_items: 22753
candidate_items_warm: 22753
candidates_not_seen_in_train: 0
evaluable_users: 11659
relevant_pairs: 24202
min_rating_relevant: 4
top_k: 10
Cek vs file resmi: {'official_file_found': True, 'official_count': 11659, 'our_count': 11659, 'identical': True, 'only_in_official': 0, 'only_in_ours': 0}

Evaluable users: 11,659


## 2. Daftar model

Satu daftar untuk empat model. Parameter Popularity, MF, TF-IDF, dan Embedding diambil dari
config dan kode terbaru di repo. Kalau Chris atau Tiffani mengubah konfigurasi final, ubah di sini
(dan catat di log). Untuk menjalankan sebagian model saja, ubah `RUN_MODELS`; baris model lain
di tabel dipertahankan dari run sebelumnya.

In [5]:
MF_PARAMS = dict(n_factors=32, n_epochs=10, learning_rate=0.01,
                 regularization=0.02, min_rating=4, random_state=SEED,
                 item_bias_weight=0.0)
TFIDF_PARAMS = dict(stop_words='english', min_df=2, max_df=0.8,
                    sublinear_tf=True, min_rating=4)   # sama dengan outputs/configs/tfidf_cbf.json
EMB_PARAMS = dict(min_rating=4)   # all-MiniLM-L6-v2, vektor dari outputs/embeddings/

MODEL_SPECS = {
    'Popularity': lambda: PopularityRecommender().fit(train, cand_set),
    'MF':         lambda: MatrixFactorizationRecommender(**MF_PARAMS).fit(train, cand_set),
    'TF-IDF':     lambda: TfidfContentRecommender(**TFIDF_PARAMS).fit(train, cand_set),
    'Embedding':  lambda: SemanticEmbeddingRecommender(**EMB_PARAMS).fit(train, cand_set),
}
RUN_MODELS = ['Popularity', 'MF', 'TF-IDF', 'Embedding']
print('Model yang dijalankan:', RUN_MODELS)

Model yang dijalankan: ['Popularity', 'MF', 'TF-IDF', 'Embedding']


## 3. Evaluasi

Cek dulu rumus metrik dengan contoh kecil yang bisa dihitung manual, lalu tiap model
dievaluasi dengan `evaluate_model(model, relevant, seen, k=10)`.
`Recorder` hanya mencatat rekomendasi supaya cek leakage di bagian 4 tidak perlu menjalankan model dua kali;
hasilnya tidak diubah.

In [6]:
sanity_check_metrics()

OK  1 hit di posisi 2 dari 2 relevan: Recall=0.5000 NDCG=0.3869 HitRate=1
OK  hit di posisi 1, 1 relevan: Recall=1.0000 NDCG=1.0000 HitRate=1
OK  tidak ada hit: Recall=0.0000 NDCG=0.0000 HitRate=0
OK  semua relevan di posisi teratas: Recall=1.0000 NDCG=1.0000 HitRate=1
OK  hit hanya di posisi 10: Recall=1.0000 NDCG=0.2891 HitRate=1
Semua contoh manual cocok.


True

In [7]:
class Recorder:
    """Membungkus model: meneruskan recommend() apa adanya dan mencatat hasilnya."""
    def __init__(self, model):
        self.model = model
        self.recs = {}
    def recommend(self, user_id, seen, k=10):
        out = self.model.recommend(user_id, seen, k)
        self.recs[user_id] = list(out)
        return out

results, recs_by_model, run_info = {}, {}, {}

for name in RUN_MODELS:
    t0 = time.time()
    try:
        model = MODEL_SPECS[name]()
        t_fit = time.time() - t0
        rec = Recorder(model)
        metrics = evaluate_model(rec, relevant, seen, k=K)
    except FileNotFoundError as e:
        print(f'[{name}] DILEWATI, file belum ada di laptop ini: {e}')
        run_info[name] = {'status': 'skipped', 'reason': str(e)}
        continue
    except Exception:
        print(f'[{name}] GAGAL:')
        traceback.print_exc()
        run_info[name] = {'status': 'failed'}
        continue
    assert metrics['n_users'] == n_users
    results[name] = metrics
    recs_by_model[name] = rec.recs
    run_info[name] = {'status': 'ok', 'fit_seconds': round(t_fit, 1),
                      'total_seconds': round(time.time() - t0, 1)}
    print(f"[{name}] n={metrics['n_users']:,}  Recall@10={metrics['Recall@10']:.4f}  "
          f"NDCG@10={metrics['NDCG@10']:.4f}  HitRate@10={metrics['HitRate@10']:.4f}  "
          f"({run_info[name]['total_seconds']} detik)")

[Popularity] n=11,659  Recall@10=0.0135  NDCG@10=0.0092  HitRate@10=0.0310  (0.9 detik)
[MF] n=11,659  Recall@10=0.0043  NDCG@10=0.0028  HitRate@10=0.0093  (555.8 detik)
[TF-IDF] n=11,659  Recall@10=0.0126  NDCG@10=0.0073  HitRate@10=0.0211  (70.4 detik)
[Embedding] DILEWATI, file belum ada di laptop ini: [Errno 2] No such file or directory: 'D:\\Kuliah\\Semester 7\\SISREK\\Proyek\\recsys-lexical-vs-semantic-g11\\outputs\\embeddings\\item_ids.npy'


## 4. Leakage check

Cek statis (split temporal, tidak ada pasangan user-item yang sama di train dan validation,
item relevan masuk kandidat, file test tidak dipakai), lalu cek rekomendasi tiap model:
tepat 10 item, tanpa duplikat, bukan item yang sudah dilihat di train, semuanya dari kandidat warm.

In [8]:
static = run_static_checks(train, val, relevant, cand_set, root=str(ROOT))
print('Cek statis:', static)

leak_rows = []
for name, recs in recs_by_model.items():
    wrong_len = sum(len(r) != K for r in recs.values())
    dup = sum(len(set(r)) != len(r) for r in recs.values())
    try:
        check_recommendations(recs, cand_set, seen)
        seen_or_outside = 0
    except LeakageError as e:
        seen_or_outside = 1
        print(f'[{name}] {e}')
    leak_rows.append({'Model': name, 'Users dicek': len(recs),
                      'Panjang != 10': wrong_len, 'Item ganda': dup,
                      'Item seen/non-kandidat': seen_or_outside})

leak_df = pd.DataFrame(leak_rows)
display(leak_df)
violations = int(leak_df[['Panjang != 10', 'Item ganda', 'Item seen/non-kandidat']].to_numpy().sum())
print('Total pelanggaran:', violations)
assert violations == 0, 'Ada pelanggaran leakage, jangan simpan hasil sebelum diperbaiki.'

Cek statis: {'temporal_split': True, 'no_pair_overlap': True, 'relevant_in_candidates': True, 'test_unused': True}


,Model,Users dicek,Panjang != 10,Item ganda,Item seen/non-kandidat
0,Popularity,11659,0,0,0
1,MF,11659,0,0,0
2,TF-IDF,11659,0,0,0


Total pelanggaran: 0


## 5. Tabel dan simpan

Satu tabel untuk semua model. `Item_unik_top10` = jumlah item berbeda di seluruh daftar top-10
(petunjuk kalau sebuah model memberi rekomendasi yang hampir sama ke semua user).
Kalau ada model yang belum dijalankan, barisnya diambil dari file sebelumnya (kalau ada).

In [10]:
new_rows = []
for name, m in results.items():
    unique_items = len({i for r in recs_by_model[name].values() for i in r})
    new_rows.append({'Model': name, 'n_users': m['n_users'],
                     'Recall@10': m['Recall@10'], 'NDCG@10': m['NDCG@10'],
                     'HitRate@10': m['HitRate@10'], 'Item_unik_top10': unique_items})
new_df = pd.DataFrame(new_rows)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = METRICS_DIR / 'validation_results.csv'
if csv_path.exists():
    old = pd.read_csv(csv_path)
    old = old[~old['Model'].isin(new_df['Model'])]
    table = pd.concat([old, new_df], ignore_index=True)
else:
    table = new_df

order = {n: i for i, n in enumerate(MODEL_SPECS)}
table = table.sort_values('Model', key=lambda s: s.map(order)).reset_index(drop=True)
table.to_csv(csv_path, index=False)
print('Tersimpan di', csv_path.relative_to(ROOT))
display(table.round(4))

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Tersimpan di outputs\metrics\validation_results.csv


,Model,n_users,Recall@10,NDCG@10,HitRate@10,Item_unik_top10
0,Popularity,11659,0.0135,0.0092,0.0310,18
1,MF,11659,0.0043,0.0028,0.0093,3620
2,TF-IDF,11659,0.0126,0.0073,0.0211,16093


## 6. File Popularity

Angka Popularity disimpan ke `outputs/metrics/popularity_val.json` (format sama dengan file metrik model lain).

In [11]:
if 'Popularity' in results:
    m = results['Popularity']
    pop_json = {'Recall@10': m['Recall@10'], 'NDCG@10': m['NDCG@10'],
                'HitRate@10': m['HitRate@10'], 'n_eval_users': m['n_users']}
    (METRICS_DIR / 'popularity_val.json').write_text(
        json.dumps(pop_json, indent=2), encoding='utf-8')
    print('Tersimpan: outputs/metrics/popularity_val.json')
    print(pop_json)
else:
    print('Popularity tidak dijalankan pada run ini.')

Tersimpan: outputs/metrics/popularity_val.json
{'Recall@10': 0.01349850373292779, 'NDCG@10': 0.009222040875545047, 'HitRate@10': 0.03104897504074106, 'n_eval_users': 11659}


## 7. Log run

Seed, parameter, jumlah user, dan versi library dicatat supaya hasilnya bisa dijalankan ulang.

In [12]:
import sklearn
LOGS_DIR.mkdir(parents=True, exist_ok=True)
run_log = {
    'run_time': datetime.datetime.now().isoformat(timespec='seconds'),
    'seed': SEED, 'k': K, 'split': 'validation', 'n_evaluable_users': n_users,
    'protocol': counts,
    'params': {'MF': MF_PARAMS, 'TF-IDF': TFIDF_PARAMS, 'Embedding': EMB_PARAMS},
    'run_models': RUN_MODELS, 'run_info': run_info,
    'leakage_violations': violations,
    'environment': {'python': sys.version.split()[0], 'numpy': np.__version__,
                    'pandas': pd.__version__, 'scikit-learn': sklearn.__version__},
}
(LOGS_DIR / 'validation_eval_run.json').write_text(
    json.dumps(run_log, indent=2, default=str), encoding='utf-8')
print('Tersimpan: outputs/logs/validation_eval_run.json')

Tersimpan: outputs/logs/validation_eval_run.json
